In [1]:
import os
import sys
import numpy as np

sys.path.append('./numba_code')
from numba_code.solver_accelerated import (
    calculate_band_structure_numba,
    calculate_nonhermitian_bands_numba,
)

sys.path.append('./original_code')
from original_code.params import get_params
from original_code.config import get_config
from original_code.plot import plot_results, plot_nonhermitian_bands


params = get_params()
DELTA_VALUES = params['DELTA_VALUES']
PHI1_VALUES = params['PHI1_VALUES']
PHI2_VALUES = params['PHI2_VALUES']
ALPHA_VALUES = params['ALPHA_VALUES']
BETA_VALUES = params['BETA_VALUES']
EPSILON0_VALUES = params['EPSILON0_VALUES']
EPSILON0_TILDE_VALUES = params['EPSILON0_TILDE_VALUES']

config = get_config()
L_TOTAL = config["L_TOTAL"]
L_A = config["L_A"]
L_F = config["L_F"]
OMEGA_MAX = config["OMEGA_MAX"]
OMEGA_STEPS = config["OMEGA_STEPS"]
TOLERANCE = config["TOLERANCE"] 

# Generate Dataset
---

In [ ]:
from tqdm.contrib.itertools import product

all_param_combinations = product(
    DELTA_VALUES, PHI1_VALUES, PHI2_VALUES, ALPHA_VALUES,
    BETA_VALUES, EPSILON0_VALUES, EPSILON0_TILDE_VALUES
)

c:\Users\Matt\Desktop\research\lin_research\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
for i, param_combination in enumerate(all_param_combinations):

    path = os.path.join('./data', str(i))
    
    if isinstance(param_combination[-2], complex):
        omega_vals, q_vals = calculate_nonhermitian_bands_numba(
            *param_combination, *list(config.values())
        )
        plot_nonhermitian_bands(omega_vals, q_vals, path)
        np.savez_compressed(file=f"./numpy_arrays/{i}", omega_values=omega_vals, q_values=q_vals)

    else:
        k_points, omega_points = calculate_band_structure_numba(
            *param_combination, *list(config.values()), save=True
        )
        plot_results(k_points, omega_points, path)
        np.savez_compressed(file=f"./numpy_arrays/{i}", k_values=k_points, omega_values=omega_points)
    
    i += 1